<a href="https://colab.research.google.com/github/AhalaAyyalas/MachineLearning/blob/main/FPv1_parts123.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PART1


In [1]:
pip install requests

In [2]:
"""
Telugu Audio Transcription using Sarvam AI (Saaras V3)
=======================================================
Best-in-class transcription for Telugu and all 22 Indian languages.
Powered by Sarvam's Saaras V3 model — built specifically for India.

SETUP (run once):
    pip install requests

GET YOUR FREE API KEY (no credit card needed):
    1. Go to https://dashboard.sarvam.ai
    2. Sign up for free
    3. Copy your API key and paste it below

USAGE:
    python telugu_transcribe_sarvam.py                       # transcribes your_audio.mp4
    python telugu_transcribe_sarvam.py path/to/myfile.mp3    # any audio file
    python telugu_transcribe_sarvam.py --folder ./recordings # batch transcribe folder
    python telugu_transcribe_sarvam.py --translate           # transcribe + translate to English

SUPPORTED FORMATS: .mp3, .mp4, .wav, .m4a, .ogg, .flac, .webm
MAX FILE SIZE: 25 MB per file
"""

import requests
import os
import sys
import argparse
import json
from datetime import datetime
from pathlib import Path


# ─── CONFIGURATION ────────────────────────────────────────────────────────────

# Paste your free API key from https://dashboard.sarvam.ai
API_KEY = "sk_1hdhhy4e_KGjo5jcZFHL3PQrGv7c9yw47"

# Or set environment variable: export SARVAM_API_KEY=your_key

SARVAM_STT_URL = "https://api.sarvam.ai/speech-to-text"

SUPPORTED_FORMATS = {
    ".mp3": "audio/mpeg",
    ".mp4": "audio/mp4",
    ".wav": "audio/wav",
    ".m4a": "audio/mp4",
    ".ogg": "audio/ogg",
    ".flac": "audio/flac",
    ".webm": "audio/webm",
}


# ─── CORE TRANSCRIPTION FUNCTION ──────────────────────────────────────────────

def transcribe_telugu(audio_path: str, translate: bool = False, save_txt: bool = True) -> str:
    """
    Transcribe a Telugu audio file using Sarvam AI's Saaras V3 model.

    Args:
        audio_path : Path to the audio/video file
        translate  : If True, also translates output to English
        save_txt   : If True, saves transcription to a .txt file

    Returns:
        Transcribed text string
    """
    audio_path = Path(audio_path)

    if not audio_path.exists():
        print(f"❌ File not found: {audio_path}")
        sys.exit(1)

    ext = audio_path.suffix.lower()
    if ext not in SUPPORTED_FORMATS:
        print(f"❌ Unsupported format: {ext}")
        print(f"   Supported: {', '.join(SUPPORTED_FORMATS.keys())}")
        sys.exit(1)

    # Check file size (Sarvam limit: 25MB)
    file_size_mb = audio_path.stat().st_size / (1024 * 1024)
    if file_size_mb > 25:
        print(f"❌ File too large: {file_size_mb:.1f} MB (max 25 MB)")
        print("   Tip: Split the file using ffmpeg:")
        print("   ffmpeg -i input.mp4 -t 600 -c copy part1.mp4")
        sys.exit(1)

    # Resolve API key
    api_key = API_KEY if API_KEY != "YOUR_SARVAM_API_KEY_HERE" else os.getenv("SARVAM_API_KEY")
    if not api_key:
        print("❌ No API key found!")
        print("   Get your FREE key at: https://dashboard.sarvam.ai")
        print("   Then set it in the script (API_KEY = '...') or via:")
        print("   export SARVAM_API_KEY=your_key")
        sys.exit(1)

    print(f"📂 File     : {audio_path.name} ({file_size_mb:.2f} MB)")
    print(f"🤖 Model    : Sarvam Saaras V3 (Telugu)")
    print(f"⏳ Transcribing...")

    mime_type = SUPPORTED_FORMATS[ext]

    # Sarvam STT API request
    with open(audio_path, "rb") as f:
        response = requests.post(
            SARVAM_STT_URL,
            headers={"api-subscription-key": api_key},
            files={"file": (audio_path.name, f, mime_type)},
            data={
                "language_code": "te-IN",       # Telugu
                "model": "saaras:v3",            # Best model
                "with_timestamps": "true",       # Include word-level timestamps
                "with_disfluencies": "false",    # Clean output (remove ums/ahs)
            }
        )

    if response.status_code != 200:
        print(f"❌ API Error {response.status_code}: {response.text}")
        sys.exit(1)

    data = response.json()
    transcript = data.get("transcript", "").strip()

    print("\n" + "=" * 60)
    print("✅ TRANSCRIPTION (Telugu)")
    print("=" * 60)
    print(transcript)
    print("=" * 60)

    # Show word-level timestamps if available
    words = data.get("words", [])
    if words:
        print("\n📋 WORD-LEVEL TIMESTAMPS:")
        print("-" * 40)
        for w in words:
            word  = w.get("word", "")
            start = w.get("start", 0)
            end   = w.get("end", 0)
            print(f"  [{start:.2f}s → {end:.2f}s]  {word}")

    # Optional: Translate to English
    english_translation = None
    if translate and transcript:
        print("\n⏳ Translating to English...")
        trans_response = requests.post(
            "https://api.sarvam.ai/translate",
            headers={
                "api-subscription-key": api_key,
                "Content-Type": "application/json"
            },
            json={
                "input": transcript,
                "source_language_code": "te-IN",
                "target_language_code": "en-IN",
                "speaker_gender": "Female",
                "mode": "formal"
            }
        )
        if trans_response.status_code == 200:
            english_translation = trans_response.json().get("translated_text", "")
            print("\n🌐 ENGLISH TRANSLATION:")
            print("-" * 40)
            print(english_translation)

    # Save to .txt file
    if save_txt:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = audio_path.parent / f"{audio_path.stem}_transcription_{timestamp}.txt"

        with open(out_path, "w", encoding="utf-8") as f:
            f.write("Telugu Audio Transcription\n")
            f.write(f"File  : {audio_path.name}\n")
            f.write(f"Model : Sarvam Saaras V3 (te-IN)\n")
            f.write(f"Date  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 60 + "\n\n")
            f.write("TELUGU TRANSCRIPT\n")
            f.write("-" * 40 + "\n")
            f.write(transcript + "\n\n")

            if english_translation:
                f.write("ENGLISH TRANSLATION\n")
                f.write("-" * 40 + "\n")
                f.write(english_translation + "\n\n")

            if words:
                f.write("WORD-LEVEL TIMESTAMPS\n")
                f.write("-" * 40 + "\n")
                for w in words:
                    f.write(f"[{w.get('start', 0):.2f}s → {w.get('end', 0):.2f}s]  {w.get('word', '')}\n")

        print(f"\n💾 Saved to : {out_path}")

    return transcript


# ─── BATCH FUNCTION ───────────────────────────────────────────────────────────

def batch_transcribe(folder_path: str, translate: bool = False):
    """Transcribe all supported audio files in a folder."""
    folder = Path(folder_path)
    files  = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED_FORMATS]

    if not files:
        print(f"No supported audio files found in: {folder_path}")
        return

    print(f"Found {len(files)} audio file(s) in '{folder_path}'\n")
    all_results = {}

    for f in sorted(files):
        print(f"\n{'─' * 50}")
        print(f"Processing: {f.name}")
        text = transcribe_telugu(str(f), translate=translate, save_txt=True)
        all_results[f.name] = text

    # Save combined batch summary
    summary_path = folder / f"batch_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("Batch Telugu Transcription — Sarvam AI\n")
        f.write(f"Date  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Files : {len(all_results)}\n")
        f.write("=" * 60 + "\n\n")
        for fname, text in all_results.items():
            f.write(f"FILE: {fname}\n")
            f.write("-" * 40 + "\n")
            f.write(text + "\n\n")

    print(f"\n✅ Batch summary saved to: {summary_path}")


# ─── CLI ENTRY POINT ──────────────────────────────────────────────────────────


In [3]:
transcribed = transcribe_telugu("srikaivalya.mp4", translate=False, save_txt=False)

📂 File     : srikaivalya.mp4 (0.24 MB)
🤖 Model    : Sarvam Saaras V3 (Telugu)
⏳ Transcribing...

✅ TRANSCRIPTION (Telugu)
శ్రీ కైవల్య పదంబు చేరుటకునై చింతించెదన్ లోక రక్షైకారంభకు భక్త పాలన కళా సంరంభకున్ దానవోద్రేక స్తంభకు కేళిలోల విలస దృగ్జాల సంభూత నానా కంజాత భవాండ కుంభకు మహానందాంగనాడింభకున్


# PART2

In [4]:
#@title Installing the libraries
!pip install chandassu regex --break-system-packages
!pip install openai-whisper
!whisper srikaivalya.mp4 --language Telugu --task transcribe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0851af7a5d53fa1adcbd59236668dee01f84334203db9414559666e28dacd3e0
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
100%|█████████████████████████████████████| 1.51G/1.51G [00:17<00:00, 91.8MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
[00:00.000 --> 00:27.280]  శ్రికఈ వల్య పదంబు ఛే రుటకునై చింతిం చేదం లోక రక్షయి కారం భకు భకు భక్తపాలన కలా సంరం భకున్

In [5]:
#@title

!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 42.7 MB/s eta 0:00:00


In [6]:
#@title U,I classification
from chandassu.telugu.laghuvu_guruvu import LaghuvuGuruvu

data = transcribed
lg = LaghuvuGuruvu(data=data).generate()

# Returns list of (aksharam, symbol) tuples
# symbol is "U" for Guruvu, "|" for Laghuvu
print(lg)

# Converting to U/I notation:
sequence = "".join("U" if s == "U" else "I" for _, s in lg)
print(sequence)  # IIUUI... etc.

[('శ్రీ', 'U'), ('కై', 'U'), ('వ', 'U'), ('ల్య', '|'), ('ప', '|'), ('దం', 'U'), ('బు', '|'), ('చే', 'U'), ('రు', '|'), ('ట', '|'), ('కు', '|'), ('నై', 'U'), ('చిం', 'U'), ('తిం', 'U'), ('చె', '|'), ('ద', 'U'), ('న్లో', 'U'), ('క', '|'), ('ర', 'U'), ('క్షై', 'U'), ('కా', 'U'), ('రం', 'U'), ('భ', '|'), ('కు', '|'), ('భ', 'U'), ('క్త', '|'), ('పా', 'U'), ('ల', '|'), ('న', '|'), ('క', '|'), ('ళా', 'U'), ('సం', 'U'), ('రం', 'U'), ('భ', '|'), ('కు', 'U'), ('న్దా', 'U'), ('న', '|'), ('వో', 'U'), ('ద్రే', 'U'), ('క', 'U'), ('స్తం', 'U'), ('భ', '|'), ('కు', '|'), ('కే', 'U'), ('ళి', '|'), ('లో', 'U'), ('ల', '|'), ('వి', '|'), ('ల', '|'), ('స', '|'), ('దృ', 'U'), ('గ్జా', 'U'), ('ల', '|'), ('సం', 'U'), ('భూ', 'U'), ('త', '|'), ('నా', 'U'), ('నా', 'U'), ('కం', 'U'), ('జా', 'U'), ('త', '|'), ('భ', '|'), ('వాం', 'U'), ('డ', '|'), ('కుం', 'U'), ('భ', '|'), ('కు', '|'), ('మ', '|'), ('హా', 'U'), ('నం', 'U'), ('దాం', 'U'), ('గ', '|'), ('నా', 'U'), ('డిం', 'U'), ('భ', '|'), ('కున్', 'U')]
UUUIIUIUIIIUUU

# PART3

In [7]:
#@title Padyam Identification

from chandassu.telugu.laghuvu_guruvu import LaghuvuGuruvu
from chandassu.telugu.padya_bhedam import check_padyam

padyam = data

# Step 1: Generate Laghuvu/Guruvu sequence
lg_data = LaghuvuGuruvu(data=padyam).generate()

# Step 2a: Print U/I sequence
print("=== U / I Sequence ===")
for aksharam, symbol in lg_data:
    print(f"  {aksharam:8s} {'U' if symbol == 'U' else 'I'}")

compact = "".join("U" if s == "U" else "I" for _, s in lg_data)
print(f"\nCompact: {compact}")

# Step 2b: Check against a specific chandas type
TYPES = [
    "vutpalamaala",
    "champakamaala",
    "mattebhamu",
    "saardulamu",
    "kandamu",
    "aataveladi",
    "teytageethi",
    "seesamu",
]

print("\n=== Check specific type ===")
result = check_padyam(
    lg_data=lg_data,
    type="vutpalamaala",   # change this to the type you want
    return_micro_score=True,
    verbose=False,          # set True for step-by-step trace
)
print(f"Overall score : {result['chandassu_score']:.0%}")
for criterion, score in result["micro_score"].items():
    bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
    print(f"  {criterion:<18s} {bar}  {score:.0%}")

# Step 2c: Auto-detect best matching chandas
print("\n=== Auto-detect best match ===")
scores = {}
for t in TYPES:
    r = check_padyam(lg_data=lg_data, type=t, return_micro_score=False, verbose=False)
    scores[t] = r["chandassu_score"]

best = max(scores, key=scores.get)
for t, s in sorted(scores.items(), key=lambda x: -x[1]):
    marker = "  <-- best match" if t == best else ""
    print(f"  {t:<20s} {s:.0%}{marker}")

=== U / I Sequence ===
  శ్రీ     U
  కై       U
  వ        U
  ల్య      I
  ప        I
  దం       U
  బు       I
  చే       U
  రు       I
  ట        I
  కు       I
  నై       U
  చిం      U
  తిం      U
  చె       I
  ద        U
  న్లో     U
  క        I
  ర        U
  క్షై     U
  కా       U
  రం       U
  భ        I
  కు       I
  భ        U
  క్త      I
  పా       U
  ల        I
  న        I
  క        I
  ళా       U
  సం       U
  రం       U
  భ        I
  కు       U
  న్దా     U
  న        I
  వో       U
  ద్రే     U
  క        U
  స్తం     U
  భ        I
  కు       I
  కే       U
  ళి       I
  లో       U
  ల        I
  వి       I
  ల        I
  స        I
  దృ       U
  గ్జా     U
  ల        I
  సం       U
  భూ       U
  త        I
  నా       U
  నా       U
  కం       U
  జా       U
  త        I
  భ        I
  వాం      U
  డ        I
  కుం      U
  భ        I
  కు       I
  మ        I
  హా       U
  నం       U
  దాం      U
  గ        I
  నా       U
  డిం      U
  భ        I
  

# New Section